# SNR / SINR analysis — over-the-air FL with unreliable clients

This notebook checks whether the following closed-form formulas are correct, and computes
them across all the saved experiment runs.

**As originally proposed:**

$$
\text{SNR}_{dB} = 10 \log_{10}\left( \frac{\left((n - N_{malicious}) \cdot \alpha_{user} \cdot w\right)^2}{P_{noise}} \right)
$$

$$
\text{SINR}_{dB} = 10 \log_{10}\left( \frac{\left((n - N_{malicious}) \cdot \alpha_{user} \cdot w\right)^2}{P_{noise} + \left(w_{malicious} \cdot N_{malicious} \cdot \alpha_{malicious}\right)^2} \right)
$$

where `n` = total users, `N_malicious` = number of malicious users, `w` / `w_malicious` are
per-user amplitudes, and `α_user` / `α_malicious` are channel gains (unit gain, i.e. `α = 1`,
since no CSI/transmit-power data is available for this system).

**The issue:** the malicious term squares the *summed* amplitude
`(N_malicious · α_malicious · w_malicious)²`, which implicitly assumes the malicious users
combine *coherently*. But they're independent attackers — independent sources add *power*,
not amplitude, so the correct term is `N_malicious · (α_malicious · w_malicious)²` (sum of
squares, not square of the sum). The reliable-user numerator is fine as written — those
updates are legitimately meant to combine coherently, which is the whole point of
over-the-air aggregation.

This notebook computes and compares four variants of the formula, and cross-checks against
the exact per-vector calculation (no scalar `w` approximation at all), so you can see exactly
how much each fix moves the number:

1. **as-given** — the formula exactly as originally written (coherent malicious term, no noise subtracted from numerator)
2. **+ incoherent malicious** — fixes only the malicious term
3. **+ noise-subtracted numerator** — also subtracts `P_noise` from the numerator, since noise is baked into the raw observed `w` before it's ever measured
4. **exact (vector-based)** — no scalar approximation: computes `‖Σ w_k‖²/d` directly from the saved per-user weight tensors

In [1]:
import math
import os

import numpy as np
import torch

def flatten(state_dict):
    return torch.cat([v.flatten().float() for v in state_dict.values()])

def load_locals(run_dir, suffix=""):
    first_path = os.path.join(run_dir, f"first_w_locals{suffix}.pt")
    latest_path = os.path.join(run_dir, f"latest_w_locals{suffix}.pt")
    first = torch.load(first_path, map_location="cpu", weights_only=False)
    latest = torch.load(latest_path, map_location="cpu", weights_only=False)
    return [flatten(sd) for sd in first], [flatten(sd) for sd in latest]

print("Imports OK")

Imports OK


## Config

`RUNS` lists every run folder with its noise std (confirmed: `001`/`003`/`005` = 0.01/0.03/0.05,
`_8`/`_16` = total user count). `VARIANTS` lists the two malicious-count variants and their
confirmed malicious indices (0-indexed) — found by nearest-neighbor distance from
`first_w_locals` entries to `latest_w_locals` entries in `sinr.py`.

`ALPHA_USER` / `ALPHA_MALICIOUS` default to 1.0 (unit channel gain — no CSI/transmit-power
data available for this system, confirmed earlier). Change these if per-user channel gains
become available.

In [2]:
RUNS = {
    "001_8":  0.01,
    "003_8":  0.03,
    "005_8":  0.05,
    "001_16": 0.01,
    "003_16": 0.03,
    "005_16": 0.05,
}

VARIANTS = [
    ("2-malicious", "", [4, 5]),
    ("1-malicious", "_1user", [6]),
]

ALPHA_USER = 1.0
ALPHA_MALICIOUS = 1.5

print(f"{len(RUNS)} run folders x {len(VARIANTS)} variants = {len(RUNS)*len(VARIANTS)} rows")

6 run folders x 2 variants = 12 rows


## Load data and compute the scalar building blocks

For each run/variant: `w` = per-element RMS amplitude of a typical reliable user
(`sqrt(mean of per-user power)`), `w_malicious` = same for a typical malicious user,
`N_reliable`/`N_malicious` = counts, `P_noise = N_reliable * noise_std^2` (independent
per-user noise summed incoherently over the reliable side — same principle used for the
malicious term).

In [3]:
records = []

for run, noise_std in RUNS.items():
    for variant_label, suffix, malicious_idx in VARIANTS:
        first_flat, latest_flat = load_locals(run, suffix)
        d = first_flat[0].numel()
        n = len(first_flat)
        reliable_idx = [i for i in range(n) if i not in malicious_idx]
        N_reliable = len(reliable_idx)
        N_malicious = len(malicious_idx)

        # scalar closed-form building blocks (SINR side, from first_w_locals)
        rel_powers = [(first_flat[i].norm() ** 2 / d).item() for i in reliable_idx]
        mal_powers = [(first_flat[i].norm() ** 2 / d).item() for i in malicious_idx]
        w = math.sqrt(np.mean(rel_powers))
        w_malicious = math.sqrt(np.mean(mal_powers))
        P_noise_sinr = N_reliable * (noise_std ** 2)

        # exact vector-based building blocks (SINR side)
        reliable_sum = torch.zeros_like(first_flat[0])
        for i in reliable_idx:
            reliable_sum += first_flat[i]
        P_signal_exact_raw = (reliable_sum.norm() ** 2 / d).item()
        P_interference_exact = sum((first_flat[i].norm() ** 2 / d).item() for i in malicious_idx)

        # scalar + exact building blocks (SNR side, from latest_w_locals -- malicious already excluded)
        M_latest = len(latest_flat)
        latest_powers = [(l.norm() ** 2 / d).item() for l in latest_flat]
        w_snr = math.sqrt(np.mean(latest_powers))
        P_noise_snr = M_latest * (noise_std ** 2)

        latest_sum = torch.zeros_like(latest_flat[0])
        for l in latest_flat:
            latest_sum += l
        P_signal_snr_exact_raw = (latest_sum.norm() ** 2 / d).item()

        records.append(dict(
            run=run, variant=variant_label, noise_std=noise_std,
            n=n, N_reliable=N_reliable, N_malicious=N_malicious, M_latest=M_latest,
            w=w, w_malicious=w_malicious, w_snr=w_snr,
            P_noise_sinr=P_noise_sinr, P_noise_snr=P_noise_snr,
            P_signal_exact_raw=P_signal_exact_raw, P_interference_exact=P_interference_exact,
            P_signal_snr_exact_raw=P_signal_snr_exact_raw,
        ))

print(f"Computed building blocks for {len(records)} run/variant combinations")
records[0]

Computed building blocks for 12 run/variant combinations


{'run': '001_8',
 'variant': '2-malicious',
 'noise_std': 0.01,
 'n': 8,
 'N_reliable': 6,
 'N_malicious': 2,
 'M_latest': 6,
 'w': 0.021661148354922163,
 'w_malicious': 0.030096979451568865,
 'w_snr': 0.0925792527364834,
 'P_noise_sinr': 0.0006000000000000001,
 'P_noise_snr': 0.0006000000000000001,
 'P_signal_exact_raw': 0.013708404265344143,
 'P_interference_exact': 0.001811656344216317,
 'P_signal_snr_exact_raw': 0.3054589629173279}

### Averaged amplitude `w` — normal (reliable) vs. malicious users

`w` = per-element RMS amplitude of a typical **reliable** user's update (`sqrt(mean of per-user power)`),
`w_malicious` = the same computed only over **malicious** users. These are the two scalar
building blocks the closed-form SINR/SNR formulas above are built from — one row per
run/variant combination.

In [4]:
import pandas as pd

w_matrix = pd.DataFrame([
    {
        "run": r["run"],
        "variant": r["variant"],
        "noise_std": r["noise_std"],
        "N_reliable": r["N_reliable"],
        "N_malicious": r["N_malicious"],
        "w (normal users, avg amplitude)": r["w"],
        "w_malicious (avg amplitude)": r["w_malicious"],
    }
    for r in records
]).set_index(["run", "variant"])

w_matrix

noise_std  N_reliable  N_malicious  \
run    variant                                           
001_8  2-malicious       0.01           6            2   
       1-malicious       0.01           7            1   
003_8  2-malicious       0.03           6            2   
       1-malicious       0.03           7            1   
005_8  2-malicious       0.05           6            2   
       1-malicious       0.05           7            1   
001_16 2-malicious       0.01          14            2   
       1-malicious       0.01          15            1   
003_16 2-malicious       0.03          14            2   
       1-malicious       0.03          15            1   
005_16 2-malicious       0.05          14            2   
       1-malicious       0.05          15            1   

                    w (normal users, avg amplitude)  \
run    variant                                        
001_8  2-malicious                         0.021661   
       1-malicious                         0.030240   
003_8  2-malicious                         0.037099   
       1-malicious                         0.043070   
005_8  2-malicious                         0.056684   
       1-malicious                         0.061507   
001_16 2-malicious                         0.029891   
       1-malicious                         0.036926   
003_16 2-malicious                         0.042376   
       1-malicious                         0.047421   
005_16 2-malicious                         0.059300   
       1-malicious                         0.063470   

                    w_malicious (avg amplitude)  
run    variant                                   
001_8  2-malicious                     0.030097  
       1-malicious                     0.043486  
003_8  2-malicious                     0.044157  
       1-malicious                     0.055343  
005_8  2-malicious                     0.064197  
       1-malicious                     0.072490  
001_16 2-malicious                     0.043477  
       1-malicious                     0.054215  
003_16 2-malicious                     0.053566  
       1-malicious                     0.062285  
005_16 2-malicious                     0.069500  
       1-malicious                     0.077247

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

display_df = w_matrix.reset_index()
display_df["noise_std"] = display_df["noise_std"].map(lambda v: f"{v:.2f}")
display_df["w (normal users, avg amplitude)"] = display_df["w (normal users, avg amplitude)"].map(lambda v: f"{v:.5f}")
display_df["w_malicious (avg amplitude)"] = display_df["w_malicious (avg amplitude)"].map(lambda v: f"{v:.5f}")
display_df = display_df.rename(columns={
    "w (normal users, avg amplitude)": "w (normal)",
    "w_malicious (avg amplitude)": "w (malicious)",
})

n_rows, n_cols = len(display_df), len(display_df.columns)
fig, ax = plt.subplots(figsize=(1.3 * n_cols, 0.42 * (n_rows + 1)), facecolor="#fcfcfb")
ax.axis("off")

table = ax.table(
    cellText=display_df.values,
    colLabels=display_df.columns,
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(n_cols)))
table.scale(1, 1.7)

for col in range(n_cols):
    cell = table[0, col]
    cell.set_facecolor("#2a78d6")
    cell.set_text_props(color="white", fontweight="bold")

for row in range(1, n_rows + 1):
    color = "#f4f3ef" if row % 2 == 0 else "#ffffff"
    for col in range(n_cols):
        table[row, col].set_facecolor(color)

fig.tight_layout()
fig.savefig("w_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved w_matrix.png")

Saved w_matrix.png


C:\Users\osman\AppData\Local\Temp\ipykernel_12412\2682964298.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## The four formula variants

`dB()` is a small helper. Each variant below computes both `SNR_dB` and `SINR_dB` for one
record, so we can line all four up side by side per run/variant.

In [6]:
def dB(x):
    return 10 * math.log10(x) if x > 0 else float("-inf")


def variant_1_as_given(r):
    """Exactly as originally proposed: coherent (squared-sum) malicious term, no noise
    subtracted from the numerator."""
    signal_num = (r["M_latest"] * ALPHA_USER * r["w_snr"]) ** 2
    snr = dB(signal_num / r["P_noise_snr"])

    signal_num_sinr = (r["N_reliable"] * ALPHA_USER * r["w"]) ** 2
    malicious_term = (r["w_malicious"] * r["N_malicious"] * ALPHA_MALICIOUS) ** 2  # coherent (bug)
    sinr = dB(signal_num_sinr / (r["P_noise_sinr"] + malicious_term))
    return snr, sinr


def variant_2_incoherent_malicious(r):
    """Fix #1 only: malicious users add power incoherently (independent sources)."""
    signal_num = (r["M_latest"] * ALPHA_USER * r["w_snr"]) ** 2
    snr = dB(signal_num / r["P_noise_snr"])

    signal_num_sinr = (r["N_reliable"] * ALPHA_USER * r["w"]) ** 2
    malicious_term = r["N_malicious"] * (ALPHA_MALICIOUS * r["w_malicious"]) ** 2  # incoherent (fixed)
    sinr = dB(signal_num_sinr / (r["P_noise_sinr"] + malicious_term))
    return snr, sinr


def variant_3_corrected(r):
    """Fix #1 + fix #2: incoherent malicious term, AND noise subtracted from the numerator
    (noise is baked into the raw observed w before it's ever measured)."""
    signal_num_snr = max((r["M_latest"] * ALPHA_USER * r["w_snr"]) ** 2 - r["P_noise_snr"], 0.0)
    snr = dB(signal_num_snr / r["P_noise_snr"])

    signal_num_sinr = max((r["N_reliable"] * ALPHA_USER * r["w"]) ** 2 - r["P_noise_sinr"], 0.0)
    malicious_term = r["N_malicious"] * (ALPHA_MALICIOUS * r["w_malicious"]) ** 2
    sinr = dB(signal_num_sinr / (r["P_noise_sinr"] + malicious_term))
    return snr, sinr


def variant_4_exact(r):
    """No scalar approximation at all: uses ||sum w_k||^2/d computed directly from the
    actual per-user vectors (sinr.py's method)."""
    signal_snr = max(r["P_signal_snr_exact_raw"] - r["P_noise_snr"], 0.0)
    snr = dB(signal_snr / r["P_noise_snr"])

    signal_sinr = max(r["P_signal_exact_raw"] - r["P_noise_sinr"], 0.0)
    sinr = dB(signal_sinr / (r["P_noise_sinr"] + r["P_interference_exact"]))
    return snr, sinr


print("Formula variants defined")

Formula variants defined


## Compare all four variants, every run/variant

In [7]:
METHODS = [
    ("as-given", variant_1_as_given),
    ("+incoherent", variant_2_incoherent_malicious),
    ("+noise-sub (corrected)", variant_3_corrected),
    ("exact (vector)", variant_4_exact),
]

results = []
for r in records:
    row = {"run": r["run"], "variant": r["variant"], "noise_std": r["noise_std"]}
    for name, fn in METHODS:
        snr_db, sinr_db = fn(r)
        row[f"SNR[{name}]"] = snr_db
        row[f"SINR[{name}]"] = sinr_db
    results.append(row)


def print_metric_table(metric):
    cols = ["run", "variant", "noise_std"] + [f"{metric}[{name}]" for name, _ in METHODS]
    widths = {c: max(len(c), 10) for c in cols}
    widths["run"] = 10
    widths["variant"] = 14

    header = "".join(f'{c:<{widths[c]}}' if c in ("run", "variant") else f'{c:>{widths[c]}}' for c in cols)
    print(header)
    print("-" * len(header))
    for row in results:
        line = "".join(
            f'{row[c]:<{widths[c]}}' if c in ("run", "variant")
            else (f'{row[c]:>{widths[c]}.3f}' if isinstance(row[c], float) else f'{row[c]:>{widths[c]}}')
            for c in cols
        )
        print(line)


print("=" * 20, "SNR_dB", "=" * 20)
print_metric_table("SNR")
print()
print("=" * 20, "SINR_dB", "=" * 20)
print_metric_table("SINR")

==================== SNR_dB ====================
run       variant        noise_stdSNR[as-given]SNR[+incoherent]SNR[+noise-sub (corrected)]SNR[exact (vector)]
-------------------------------------------------------------------------------------------------------------
001_8     2-malicious        0.010       27.112          27.112                     27.103             27.059
001_8     1-malicious        0.010       27.481          27.481                     27.473             27.426
003_8     2-malicious        0.030       23.948          23.948                     23.930             23.841
003_8     1-malicious        0.030       23.912          23.912                     23.895             23.787
005_8     2-malicious        0.050       23.498          23.498                     23.478             23.379
005_8     1-malicious        0.050       23.505          23.505                     23.486             23.367
001_16    2-malicious        0.010       27.470          27.470        

## Plot: SINR across the four methods, 2-malicious variant

Small multiples by user count, one line per method, so you can see how far "as-given"
drifts from "exact" and confirm each fix moves it the right direction.

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

METHOD_COLORS = {
    "as-given": "#2a78d6",                 # slot 1 blue
    "+incoherent": "#eb6834",              # slot 2 orange
    "+noise-sub (corrected)": "#1baf7a",   # slot 3 aqua
    "exact (vector)": "#eda100",           # slot 4 yellow
}
GRIDLINE, AXIS, TEXT_SECONDARY, TEXT_MUTED = "#e1e0d9", "#c3c2b7", "#52514e", "#898781"

variant_rows = [row for row in results if row["variant"] == "2-malicious"]
panels = sorted({r["run"].split("_")[1] for r in variant_rows}, key=int)  # "8", "16"

fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 4.2), sharey=True)
if len(panels) == 1:
    axes = [axes]

for ax, n_users in zip(axes, panels):
    panel_rows = [r for r in variant_rows if r["run"].endswith(f"_{n_users}")]
    panel_rows = sorted(panel_rows, key=lambda r: r["run"])
    x = np.arange(len(panel_rows))
    run_labels = [r["run"] for r in panel_rows]

    for name, _ in METHODS:
        y = [r[f"SINR[{name}]"] for r in panel_rows]
        ax.plot(x, y, color=METHOD_COLORS[name], linewidth=2.2, marker="o",
                 markersize=7, markeredgecolor="white", markeredgewidth=1.4, zorder=3)

    ax.set_title(f"{n_users} users", fontsize=12, fontweight="bold", loc="left", color="#0b0b0b")
    ax.set_xticks(x)
    ax.set_xticklabels(run_labels, color=TEXT_SECONDARY)
    ax.set_xlabel("run (increasing noise std →)", color=TEXT_MUTED, fontsize=9)
    ax.grid(axis="y", color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(AXIS)
    ax.tick_params(axis="both", colors=TEXT_MUTED, length=0)
    ax.margins(x=0.15)

axes[0].set_ylabel("SINR (dB)", color=TEXT_SECONDARY, fontsize=10)

legend_handles = [
    Line2D([0], [0], color=color, linewidth=2.2, marker="o", markersize=6,
           markeredgecolor="white", markeredgewidth=1.2, label=name)
    for name, color in METHOD_COLORS.items()
]
fig.legend(handles=legend_handles, loc="lower center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, -0.06), fontsize=9, labelcolor=TEXT_SECONDARY)
fig.suptitle("SINR: formula variants vs. exact (2-malicious)", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("sinr_formula_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved sinr_formula_comparison.png")

Saved sinr_formula_comparison.png


C:\Users\osman\AppData\Local\Temp\ipykernel_12412\2738385364.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Plot: SNR across the same methods, 2-malicious variant

Same layout as the SINR plot above. SNR has no malicious term, so it's unaffected by
`ALPHA_MALICIOUS` — only the noise-subtraction fix moves it at all.

In [9]:
fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 4.2), sharey=True)
if len(panels) == 1:
    axes = [axes]

for ax, n_users in zip(axes, panels):
    panel_rows = [r for r in variant_rows if r["run"].endswith(f"_{n_users}")]
    panel_rows = sorted(panel_rows, key=lambda r: r["run"])
    x = np.arange(len(panel_rows))
    run_labels = [r["run"] for r in panel_rows]

    for name, _ in METHODS:
        y = [r[f"SNR[{name}]"] for r in panel_rows]
        ax.plot(x, y, color=METHOD_COLORS[name], linewidth=2.2, marker="o",
                 markersize=7, markeredgecolor="white", markeredgewidth=1.4, zorder=3)

    ax.set_title(f"{n_users} users", fontsize=12, fontweight="bold", loc="left", color="#0b0b0b")
    ax.set_xticks(x)
    ax.set_xticklabels(run_labels, color=TEXT_SECONDARY)
    ax.set_xlabel("run (increasing noise std →)", color=TEXT_MUTED, fontsize=9)
    ax.grid(axis="y", color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(AXIS)
    ax.tick_params(axis="both", colors=TEXT_MUTED, length=0)
    ax.margins(x=0.15)

axes[0].set_ylabel("SNR (dB)", color=TEXT_SECONDARY, fontsize=10)

legend_handles = [
    Line2D([0], [0], color=METHOD_COLORS[name], linewidth=2.2, marker="o", markersize=6,
           markeredgecolor="white", markeredgewidth=1.2, label=name)
    for name, _ in METHODS
]
fig.legend(handles=legend_handles, loc="lower center", ncol=len(METHODS), frameon=False,
           bbox_to_anchor=(0.5, -0.06), fontsize=9, labelcolor=TEXT_SECONDARY)
fig.suptitle("SNR: formula variants vs. exact (2-malicious)", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("snr_formula_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved snr_formula_comparison.png")

C:\Users\osman\AppData\Local\Temp\ipykernel_12412\2883380318.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved snr_formula_comparison.png


## Plot: SINR & SNR combined into one PNG

Same small-multiples layout as the two plots above, but SINR and SNR drawn as two colored
lines in the same panel (blue = SINR, orange = SNR — same palette slots `sinr.py` already
uses), so the post-defense-vs-pre-defense gap is visible in a single image instead of two.

In [10]:
METRIC_COLORS = {
    "SINR": "#2a78d6",  # slot 1 blue
    "SNR": "#eb6834",   # slot 2 orange
}

fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 4.2), sharey=True)
if len(panels) == 1:
    axes = [axes]

for ax, n_users in zip(axes, panels):
    panel_rows = [r for r in variant_rows if r["run"].endswith(f"_{n_users}")]
    panel_rows = sorted(panel_rows, key=lambda r: r["run"])
    x = np.arange(len(panel_rows))
    run_labels = [r["run"] for r in panel_rows]

    for metric, color in METRIC_COLORS.items():
        for name, _ in METHODS:
            y = [r[f"{metric}[{name}]"] for r in panel_rows]
            ax.plot(x, y, color=color, linewidth=2.2, marker="o",
                     markersize=7, markeredgecolor="white", markeredgewidth=1.4, zorder=3)

    ax.set_title(f"{n_users} users", fontsize=12, fontweight="bold", loc="left", color="#0b0b0b")
    ax.set_xticks(x)
    ax.set_xticklabels(run_labels, color=TEXT_SECONDARY)
    ax.set_xlabel("run (increasing noise std →)", color=TEXT_MUTED, fontsize=9)
    ax.grid(axis="y", color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(AXIS)
    ax.tick_params(axis="both", colors=TEXT_MUTED, length=0)
    ax.margins(x=0.15)

axes[0].set_ylabel("dB", color=TEXT_SECONDARY, fontsize=10)

legend_handles = [
    Line2D([0], [0], color=color, linewidth=2.2, marker="o", markersize=6,
           markeredgecolor="white", markeredgewidth=1.2, label=metric)
    for metric, color in METRIC_COLORS.items()
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.5, -0.06), fontsize=9, labelcolor=TEXT_SECONDARY)
method_names = ", ".join(name for name, _ in METHODS)
fig.suptitle(f"SINR & SNR ({method_names}, 2-malicious)", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("sinr_snr_combined.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved sinr_snr_combined.png")

Saved sinr_snr_combined.png


C:\Users\osman\AppData\Local\Temp\ipykernel_12412\94391827.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Conclusion

**The originally proposed formula is not correct as written**, but the fix is small and the
result is a sensible closed-form approximation once corrected.

- The **SNR formula** was already fine — it has no malicious term to get wrong.
- The **SINR formula**'s malicious term `(w_malicious · N_malicious · α_malicious)²` treats
  independent attackers as if they combine coherently. Since they're independent, their
  *powers* add, not their amplitudes — the correct term is
  `N_malicious · (α_malicious · w_malicious)²`. Run the tables above and you'll see this fix
  alone moves `SINR[as-given]` noticeably closer to `SINR[exact (vector)]`.
- Subtracting `P_noise` from the numerator (fix #2) accounts for noise being baked into the
  raw observed `w` before it's ever measured — a smaller additional correction.
- Even after both fixes, the closed-form result sits slightly above the exact vector-based
  value, because `((n-N_malicious)·w)²` assumes every reliable user's weight vector points in
  *exactly* the same direction (perfect coherent alignment). Real per-user vectors are close
  but not perfectly aligned, so `‖Σw_k‖² ≤ (Σ‖w_k‖)²` — the closed form is an upper-bound
  idealization, useful for an analytical formula in a paper, but the exact number should come
  from `sinr.py`'s direct vector computation.